In [1]:
import pandas as pd
import numpy as np

In [ ]:
college_data = pd.read_csv('data/raw_college_data/college_shooting_model_data.csv')

In [ ]:
# Remove rows with null values
college_data_no_na = college_data.dropna(subset=["GP"])

In [ ]:
# Drop empty columns
college_data_cleaned = college_data_no_na.drop(columns=['Column 2', 'Column 3', 'Column 4', 'Column 5', 'Column 6',
       'Column 7', 'Column 8', 'Column 9', 'Column 10', 'Column 11',
       'Column 12', 'Column 13', 'Column 14'])

In [ ]:
# Replace double single quotes with double quotes in the Height and Wingspan columns
college_data_cleaned["Height (Inches)"] = college_data_cleaned["Height (Inches)"].str.replace("''", '"')
college_data_cleaned["Wingspan (Inches)"] = college_data_cleaned["Wingspan (Inches)"].str.replace("''", '"')

In [ ]:
# Convert height and wingspan from feet and inches to inches
def to_inches(feet_inches):
    if pd.isna(feet_inches):
        return pd.NA
    feet_inches_split = feet_inches[:-1].split("'")
    if len(feet_inches_split) != 2:
        return pd.NA
    feet = int(feet_inches_split[0].strip())
    inches = float(feet_inches_split[1].strip())
    return feet * 12 + inches

In [ ]:
# Apply the conversion function to the Height and Wingspan columns
college_data_cleaned["Height (Inches)"] = college_data_cleaned["Height (Inches)"].apply(to_inches)
college_data_cleaned["Wingspan (Inches)"] = college_data_cleaned["Wingspan (Inches)"].apply(to_inches)

In [ ]:
# Create a new DataFrame for feature data
feature_data = college_data_cleaned.copy()

In [ ]:
# Create new features
feature_data["C&S_3PT_PPS"] = feature_data["C&S_3PT%"] * 3
feature_data["OTD_ATT_PG"] = feature_data["OTD_ATT"] / feature_data["GP"]
feature_data["C&S_3PT_ATT_PG"] = feature_data["C&S_3PT_ATT"] / feature_data["GP"]
feature_data["Shot_diet_OTD"] = feature_data["OTD_ATT"] / feature_data["FG_ATT"]
feature_data["Shot_diet_C&S"] = feature_data["C&S_3PT_ATT"] / feature_data["FG_ATT"]
feature_data["C&S_Guarded_prop"] = feature_data["C&S_3PT_G_ATT"] / feature_data["C&S_3PT_ATT"]
feature_data["Length"] = feature_data["Wingspan (Inches)"] - feature_data["Height (Inches)"]

In [ ]:
# Rename column for clarity
feature_data.rename(columns={"Height (Inches)": "Height"}, inplace=True)

In [ ]:
# Select relevant columns for modeling
feature_data_filtered = feature_data[["Name", "C&S_3PT_PPS", "OTD_PPS", "FT%", "C&S_3PT_ATT_PG", "OTD_ATT_PG", "Shot_diet_C&S", "Shot_diet_OTD", "C&S_Guarded_prop", "Height", "Length", "Age"]]

In [ ]:
# Save the cleaned and feature-engineered data to a new CSV file
feature_data_filtered.to_csv('data/cleaned_college_data/college_feature_data.csv', index=False)